<a href="https://colab.research.google.com/github/NagaRaju1991/google_colab_notebooks/blob/fsds_course/01_huggingface_Basic_introduction_to_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤗 Getting Started with Hugging Face Transformers

## A Comprehensive Hands-on Transformers - Seesion complete

---

### What is Hugging Face?

**Hugging Face** is an AI company that has become the central hub for machine learning, particularly in Natural Language Processing (NLP). They provide:

1. **Transformers Library**: An open-source library that provides thousands of pre-trained models for various NLP tasks
2. **Model Hub**: A platform hosting over 200,000+ pre-trained models shared by the community
3. **Datasets Library**: Easy access to thousands of datasets for ML tasks
4. **Spaces**: A platform to host ML demos and applications

### Why Use Hugging Face?

- **Easy to Use**: Just a few lines of code to use state-of-the-art models
- **Pre-trained Models**: Access models trained on massive datasets without needing extensive compute resources
- **Flexibility**: Works with PyTorch, TensorFlow, and JAX
- **Community**: Vibrant community contributing models, datasets, and tutorials


---

---

### What is Hugging Face?

**Hugging Face** is an AI company that has become the central hub for machine learning, particularly in Natural Language Processing (NLP). They provide:

1. **Transformers Library**: An open-source library that provides thousands of pre-trained models for various NLP tasks
2. **Model Hub**: A platform hosting over 200,000+ pre-trained models shared by the community
3. **Datasets Library**: Easy access to thousands of datasets for ML tasks
4. **Spaces**: A platform to host ML demos and applications

### Why Use Hugging Face?

- **Easy to Use**: Just a few lines of code to use state-of-the-art models
- **Pre-trained Models**: Access models trained on massive datasets without needing extensive compute resources
- **Flexibility**: Works with PyTorch, TensorFlow, and JAX
- **Community**: Vibrant community contributing models, datasets, and tutorials

### What We'll Cover in This Notebook

**Part 1: Important Classes in Hugging Face Transformers**
- Core Classes Overview and Architecture
- AutoClasses (AutoTokenizer, AutoModel, AutoConfig)
- Model Classes for Different Tasks
- Configuration Classes

**Part 2: Getting Started with Pipelines**
- Sentiment Analysis
- Zero-Shot Classification
- Text Generation with Different Decoding Strategies
- Named Entity Recognition (NER)
- Question Answering

**Part 3: Working with Tokenizers**
- Loading and Using Tokenizers
- Comparing Different Tokenizers (BERT vs GPT-2)
- Padding and Truncation
- Adding Custom Tokens

---

#### SentencePiece: Unsupervised Text Tokenizer

#### Overview
**SentencePiece** is an open-source library and unsupervised text tokenizer developed by Google, primarily for neural network-based text generation systems like **LLaMA, T5, and XLNet**.

**Key Innovation**: It handles the **"open-vocabulary" problem** in NLP by breaking text into smaller **subword units** without requiring language-specific rules.

---

#### Core Capabilities

#### 1. **Language Independence**
- Treats text as **raw Unicode sequence**
- No language-specific preprocessing required

#### 2. **Lossless Tokenization**
- **▁ (underscore)** represents spaces (preserves whitespace)
- **Exact reconstruction** of original text from tokens
- Unlike regex-based tokenizers that lose formatting

#### 3. **Subword Algorithms** (Two Approaches)

| Algorithm | Direction | How it works |
|-----------|-----------|-------------|
| **BPE** | **Bottom-up** | Start with characters → merge frequent pairs |
| **Unigram LM** | **Top-down** | Start with large vocab → prune least useful tokens |

**BPE Example**:

#### 4. **Subword Regularization**
- **Multiple tokenizations** per sentence during training
- Acts as **data augmentation**
- Makes models **more robust** to tokenization variations

---

## Setup and Installation

First, let's install the required libraries.

In [1]:
# Install the required libraries
# transformers: The main Hugging Face library for working with transformer models
# torch: PyTorch deep learning framework (backend for transformers)
# sentencepiece: Required for some tokenizers (like T5, ALBERT)
# accelerate: Helps with model loading and inference optimization

!pip install -q transformers torch sentencepiece accelerate

#### What is pipeline
Raw Input → Tokenizer → Model → Post-Processing → Ready Results </br>
     
"Hello world"  → tokens → logits → predictions → "Positive sentiment"  </br>

pipeline() is Hugging Face's high-level API that wraps tokenizer + model + post-processing into a single inference function

## 1.1 Sentiment Analysis Pipeline

### What is Sentiment Analysis?

Sentiment analysis is the task of classifying text based on the emotional tone. The most common classification is:
- **POSITIVE**: Text expresses favorable sentiment
- **NEGATIVE**: Text expresses unfavorable sentiment

Some models also support **NEUTRAL** sentiment.

### How It Works

1. Text is tokenized (converted to numbers)
2. Tokens are passed through a transformer model
3. The model outputs probabilities for each class
4. The highest probability class is selected

---

# Part 1: Getting Started with Pipelines

## What are Pipelines?

**Pipelines** are Hugging Face's highest-level API. They abstract away all the complexity of:
- Loading a pre-trained model
- Loading the corresponding tokenizer
- Preprocessing input text
- Running inference
- Post-processing outputs

With just **one line of code**, you can perform complex NLP tasks!

### Available Pipeline Tasks

| Task | Description | Example Use Case |
|------|-------------|------------------|
| `sentiment-analysis` | Classify text as positive/negative | Product reviews |
| `zero-shot-classification` | Classify text into custom labels | Content categorization |
| `text-generation` | Generate text continuations | Creative writing |
| `ner` | Extract named entities | Information extraction |
| `question-answering` | Answer questions from context | Document QA |
| `summarization` | Summarize long texts | News summarization |
| `translation` | Translate between languages | Multilingual apps |

---



In [2]:
# Import necessary libraries
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

# Core transformers imports
from transformers import pipeline  # High-level API for common NLP tasks
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM  # For loading models/tokenizers
from transformers import set_seed  # For reproducibility

# Standard Python libraries
import torch

# Set seed for reproducibility in text generation
set_seed(42)

print(f"Transformers version: {__import__('transformers').__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Transformers version: 5.0.0
PyTorch version: 2.10.0+cu128
CUDA available: True


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

text = "The capital of India is Delhi and it is a beautiful city."
inputs = tokenizer(text, return_tensors="pt")

print('=='*30)
print(inputs)
print('=='*30)
with torch.no_grad():
    outputs = model(**inputs, labels=inputs["input_ids"])
    loss = outputs.loss          # This is the average cross-entropy loss
    perplexity = torch.exp(loss) # This is the perplexity

print(f"Loss: {loss.item():.4f}")
print(f"Perplexity: {perplexity.item():.2f}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

{'input_ids': tensor([[  464,  3139,   286,  3794,   318, 12517,   290,   340,   318,   257,
          4950,  1748,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Loss: 2.8675
Perplexity: 17.59


In [ ]:
# Import necessary libraries
import warnings
warnings.filterwarnings('ignore')

# Core transformers imports
from transformers import pipeline
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, AutoConfig
from transformers import set_seed

import torch

set_seed(42)

print(f"Transformers version: {__import__('transformers').__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Transformers version: 5.0.0
PyTorch version: 2.10.0+cu128
CUDA available: True


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "The capital of India is Delhi."

# Without return_tensors
output1 = tokenizer(text)
print(type(output1['input_ids']))
print(output1['input_ids'])        # → <class 'list'>

# With return_tensors="pt"
output2 = tokenizer(text, return_tensors="pt")
print(output2['input_ids'])        # → <class 'torch.Tensor'>

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

<class 'list'>
[101, 1996, 3007, 1997, 2634, 2003, 6768, 1012, 102]
tensor([[ 101, 1996, 3007, 1997, 2634, 2003, 6768, 1012,  102]])


In [ ]:
# ============================================================
# SENTIMENT ANALYSIS WITH DEFAULT MODEL
# ============================================================

# Create a sentiment analysis pipeline
# By default, this uses 'distilbert-base-uncased-finetuned-sst-2-english'
# which is a lightweight model fine-tuned on the Stanford Sentiment Treebank

sentiment_analyzer = pipeline(
    "sentiment-analysis",  # The task we want to perform
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available, else CPU
)

print(":> Sentiment analyzer loaded!")
print(f"Model: {sentiment_analyzer.model.config._name_or_path}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

:> Sentiment analyzer loaded!
Model: distilbert/distilbert-base-uncased-finetuned-sst-2-english


In [ ]:
# Define custom sentences to analyze
# We'll test various scenarios: clearly positive, clearly negative, and nuanced cases

new_sentences = [
    # Clearly positive
    "I absolutely love this product! It exceeded all my expectations.",
    "The customer service was exceptional and very helpful.",

    # Clearly negative
    "This is the worst experience I've ever had. Complete waste of money.",
    "The quality is terrible and it broke after one day.",

    # Mixed sentiments (challenging cases)
    "The food was good but the service was slow.",
    "Not bad, but I expected more for the price.",
    "It's okay, nothing special.",

    # Sarcasm (often misclassified)
    "Oh great, another meeting that could have been an email.",

    # Technical/Neutral
    "The package arrived on Tuesday."
]

# Analyze each sentence
print("\n" + "="*70)
print("SENTIMENT ANALYSIS RESULTS (Default Model)")
print("="*70 + "\n")

for sentence in new_sentences:
    # Get prediction
    print(f"result will be of the form "  , sentiment_analyzer(sentence))
    result = sentiment_analyzer(sentence)  # [0] because we're analyzing one sentence
    print(result)
    predict = result[0]
    print(predict)
    # Format output with emoji for clarity
    emoji = "😊" if predict['label'] == 'POSITIVE' else "😞"
    confidence = predict['score'] * 100  # Convert to percentage

    print(f"Text: \"{sentence[:60]}{'...' if len(sentence) > 60 else ''}\"")
    print(f"Result: {emoji} {predict['label']} (Confidence: {confidence:.1f}%)")
    print("-" * 70)


SENTIMENT ANALYSIS RESULTS (Default Model)



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


result will be of the form  [{'label': 'POSITIVE', 'score': 0.9998822212219238}]
[{'label': 'POSITIVE', 'score': 0.9998822212219238}]
{'label': 'POSITIVE', 'score': 0.9998822212219238}
Text: "I absolutely love this product! It exceeded all my expectati..."
Result: 😊 POSITIVE (Confidence: 100.0%)
----------------------------------------------------------------------
result will be of the form  [{'label': 'POSITIVE', 'score': 0.9998583793640137}]
[{'label': 'POSITIVE', 'score': 0.9998583793640137}]
{'label': 'POSITIVE', 'score': 0.9998583793640137}
Text: "The customer service was exceptional and very helpful."
Result: 😊 POSITIVE (Confidence: 100.0%)
----------------------------------------------------------------------
result will be of the form  [{'label': 'NEGATIVE', 'score': 0.9997848868370056}]
[{'label': 'NEGATIVE', 'score': 0.9997848868370056}]
{'label': 'NEGATIVE', 'score': 0.9997848868370056}
Text: "This is the worst experience I've ever had. Complete waste o..."
Result: 😞 NEGATI

---

# Part 1: Important Classes in Hugging Face Transformers

## Hugging Face Transformers Library Architecture

The Transformers library is organized into several key component categories:

```
transformers/
    |
    |-- Auto Classes (Automatic model/tokenizer loading)
    |   |-- AutoConfig
    |   |-- AutoTokenizer  
    |   |-- AutoModel
    |   |-- AutoModelForXXX (task-specific)
    |
    |-- Model Classes (Architecture implementations)
    |   |-- BertModel, GPT2Model, T5Model, etc.
    |   |-- BertForSequenceClassification, etc.
    |
    |-- Tokenizer Classes (Text preprocessing)
    |   |-- BertTokenizer, GPT2Tokenizer, etc.
    |   |-- PreTrainedTokenizer, PreTrainedTokenizerFast
    |
    |-- Configuration Classes (Model hyperparameters)
    |   |-- BertConfig, GPT2Config, etc.
    |
    |-- Pipeline Classes (High-level inference API)
    |   |-- pipeline()
    |
    |-- Training Utilities
        |-- Trainer, TrainingArguments
        |-- DataCollator classes
```

---

## 1.1 The Most Important Classes - Quick Reference

| Class Category | Class Name | Purpose | When to Use |
|---------------|------------|---------|-------------|
| **Auto Classes** | `AutoTokenizer` | Automatically loads the correct tokenizer | Always - most flexible approach |
| | `AutoModel` | Loads base model (no task head) | Feature extraction, embeddings |
| | `AutoModelForSequenceClassification` | Classification tasks | Sentiment, topic classification |
| | `AutoModelForTokenClassification` | Token-level tasks | NER, POS tagging |
| | `AutoModelForQuestionAnswering` | QA tasks | Extractive QA |
| | `AutoModelForCausalLM` | Text generation | GPT-style generation |
| | `AutoModelForSeq2SeqLM` | Sequence-to-sequence | Translation, summarization |
| | `AutoConfig` | Load model configuration | Inspect/modify model settings |
| **Tokenizers** | `PreTrainedTokenizer` | Base tokenizer class | Understanding tokenization |
| | `PreTrainedTokenizerFast` | Rust-based fast tokenizer | Production use |
| **Training** | `Trainer` | Training loop abstraction | Fine-tuning models |
| | `TrainingArguments` | Training configuration | Setting hyperparameters |
| **Pipeline** | `pipeline()` | End-to-end inference | Quick prototyping |

---

## 1.2 Auto Classes - Detailed Explanation

### What are Auto Classes?

Auto classes are **factory classes** that automatically determine and load the correct model architecture based on the model name or path. They eliminate the need to know the exact class name for each model.

### Why Use Auto Classes?

1. **Model Agnostic**: Same code works for BERT, GPT-2, T5, etc.
2. **Future Proof**: New models work without code changes
3. **Less Error Prone**: No mismatch between tokenizer and model
4. **Cleaner Code**: One consistent API across all models

### BERT-Base-Uncased in Hugging Face

### Overview
`bert-base-uncased` is a pretrained language model based on BERT (Bidirectional Encoder Representations from Transformers). It is widely used for various Natural Language Processing (NLP) tasks such as text classification, question answering, and named entity recognition.

---

### What does the name mean?

### 1. BERT
- A transformer-based model
- Processes text **bidirectionally** (left + right context)
- Designed for deep language understanding

---

### 2. Base
Indicates the model size:

- 12 Transformer layers (encoder blocks)
- Hidden size: 768
- 12 attention heads
- ~110 million parameters

---

### 3. Uncased
- Converts all text to lowercase before processing  
- Example:

In [ ]:
# ============================================================
# AUTOCONFIG - Model Configuration
# ============================================================

from transformers import AutoConfig

# AutoConfig loads the configuration (hyperparameters) of a model
# This includes: hidden_size, num_layers, num_attention_heads, vocab_size, etc.

config = AutoConfig.from_pretrained("bert-base-uncased")

print("="*60)
print("BERT-BASE-UNCASED CONFIGURATION")
print("="*60)
print(f"\nModel type: {config.model_type}")
print(f"Hidden size: {config.hidden_size}")
print(f"Number of layers: {config.num_hidden_layers}")
print(f"Number of attention heads: {config.num_attention_heads}")
print(f"Vocabulary size: {config.vocab_size}")
print(f"Max position embeddings: {config.max_position_embeddings}")
print(f"Hidden activation: {config.hidden_act}")

BERT-BASE-UNCASED CONFIGURATION

Model type: bert
Hidden size: 768
Number of layers: 12
Number of attention heads: 12
Vocabulary size: 30522
Max position embeddings: 512
Hidden activation: gelu


| Token  | ID  | Purpose                                   |
| ------ | --- | ----------------------------------------- |
| [CLS]  | 101 | Sequence representation (pooled output)   |
| [SEP]  | 102 | Sentence separator (multi-sentence input) |
| [PAD]  | 0   | Padding shorter sequences                 |
| [UNK]  | 100 | Unknown words (rare)                      |
| [MASK] | 103 | Masked LM pretraining                     |

In [ ]:
# ============================================================
# AUTOTOKENIZER - Text Preprocessing
# ============================================================

from transformers import AutoTokenizer

# AutoTokenizer automatically selects the correct tokenizer class
# For BERT: BertTokenizer, For GPT-2: GPT2Tokenizer, etc.

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("="*60)
print("AUTOTOKENIZER DEMONSTRATION")
print("="*60)

# Key tokenizer attributes
print(f"\nTokenizer class: {type(tokenizer).__name__}")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Model max length: {tokenizer.model_max_length}")
print(f"Padding token: {tokenizer.pad_token}")
print(f"Unknown token: {tokenizer.unk_token}")
print(f"Special tokens: {tokenizer.all_special_tokens}")

# Basic tokenization example
text = "Hello, how are you?"
encoded = tokenizer(text, return_tensors="pt")

print(f"\nInput text: '{text}'")
print(f"Token IDs: {encoded['input_ids'].tolist()[0]}")
print(f"Attention mask: {encoded['attention_mask'].tolist()[0]}")

AUTOTOKENIZER DEMONSTRATION

Tokenizer class: BertTokenizer
Vocabulary size: 30522
Model max length: 512
Padding token: [PAD]
Unknown token: [UNK]
Special tokens: ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']

Input text: 'Hello, how are you?'
Token IDs: [101, 7592, 1010, 2129, 2024, 2017, 1029, 102]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
# ============================================================
# AUTOMODEL - Base Model (No Task Head)
# ============================================================

from transformers import AutoModel

# AutoModel loads the base transformer without any task-specific head
# Use this for: feature extraction, embeddings, custom architectures

model = AutoModel.from_pretrained("bert-base-uncased")

print("="*60)
print("AUTOMODEL - BASE MODEL")
print("="*60)

print(f"\nModel class: {type(model).__name__}")
print(f"Number of parameters: {model.num_parameters():,}")

# Get model output (hidden states)
text = "Hello, world!"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

print(f"\nOutput type: {type(outputs).__name__}")
print(f"Last hidden state shape: {outputs.last_hidden_state.shape}")
print(f"  - Batch size: {outputs.last_hidden_state.shape[0]}")
print(f"  - Sequence length: {outputs.last_hidden_state.shape[1]}")
print(f"  - Hidden dimension: {outputs.last_hidden_state.shape[2]}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AUTOMODEL - BASE MODEL

Model class: BertModel
Number of parameters: 109,482,240

Output type: BaseModelOutputWithPoolingAndCrossAttentions
Last hidden state shape: torch.Size([1, 6, 768])
  - Batch size: 1
  - Sequence length: 6
  - Hidden dimension: 768


In [ ]:
# ============================================================
# TASK-SPECIFIC AUTO MODELS
# ============================================================

from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoModelForQuestionAnswering,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoModelForMaskedLM
)

print("="*60)
print("TASK-SPECIFIC AUTO MODEL CLASSES")
print("="*60)

task_models = {
    "AutoModelForSequenceClassification": {
        "use_case": "Sentiment analysis, text classification",
        "output": "Logits for each class",
        "example_model": "bert-base-uncased"
    },
    "AutoModelForTokenClassification": {
        "use_case": "NER, POS tagging, chunking",
        "output": "Logits for each token",
        "example_model": "dslim/bert-base-NER"
    },
    "AutoModelForQuestionAnswering": {
        "use_case": "Extractive question answering",
        "output": "Start/end position logits",
        "example_model": "distilbert-base-cased-distilled-squad"
    },
    "AutoModelForCausalLM": {
        "use_case": "Text generation (GPT-style)",
        "output": "Next token logits",
        "example_model": "gpt2"
    },
    "AutoModelForSeq2SeqLM": {
        "use_case": "Translation, summarization",
        "output": "Decoder output logits",
        "example_model": "t5-small"
    },
    "AutoModelForMaskedLM": {
        "use_case": "Fill-in-the-blank, pre-training",
        "output": "Token prediction logits",
        "example_model": "bert-base-uncased"
    }
}

for class_name, info in task_models.items():
    print(f"\n{class_name}")
    print(f"  Use case: {info['use_case']}")
    print(f"  Output: {info['output']}")
    print(f"  Example: {info['example_model']}")

TASK-SPECIFIC AUTO MODEL CLASSES

AutoModelForSequenceClassification
  Use case: Sentiment analysis, text classification
  Output: Logits for each class
  Example: bert-base-uncased

AutoModelForTokenClassification
  Use case: NER, POS tagging, chunking
  Output: Logits for each token
  Example: dslim/bert-base-NER

AutoModelForQuestionAnswering
  Use case: Extractive question answering
  Output: Start/end position logits
  Example: distilbert-base-cased-distilled-squad

AutoModelForCausalLM
  Use case: Text generation (GPT-style)
  Output: Next token logits
  Example: gpt2

AutoModelForSeq2SeqLM
  Use case: Translation, summarization
  Output: Decoder output logits
  Example: t5-small

AutoModelForMaskedLM
  Use case: Fill-in-the-blank, pre-training
  Output: Token prediction logits
  Example: bert-base-uncased


In [ ]:
# ============================================================
# PRACTICAL EXAMPLE: Loading Different Model Types
# ============================================================

print("="*60)
print("LOADING MODELS FOR DIFFERENT TASKS")
print("="*60)

# Example 1: Sequence Classification (Sentiment)
print("\n1. Loading model for Sequence Classification...")
classifier = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
print(f"   Model: {type(classifier).__name__}")
print(f"   Number of labels: {classifier.config.num_labels}")
print(f"   Labels: {classifier.config.id2label}")

# Example 2: Causal Language Model (Text Generation)
print("\n2. Loading model for Text Generation...")
generator = AutoModelForCausalLM.from_pretrained("gpt2")
print(f"   Model: {type(generator).__name__}")
print(f"   Vocabulary size: {generator.config.vocab_size}")

# Clean up to save memory
del classifier, generator
torch.cuda.empty_cache() if torch.cuda.is_available() else None

LOADING MODELS FOR DIFFERENT TASKS

1. Loading model for Sequence Classification...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

   Model: DistilBertForSequenceClassification
   Number of labels: 2
   Labels: {0: 'NEGATIVE', 1: 'POSITIVE'}

2. Loading model for Text Generation...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Model: GPT2LMHeadModel
   Vocabulary size: 50257


## 1.3 Tokenizer Classes - Deep Dive

### Types of Tokenizers

| Tokenizer Type | Description | Speed | Features |
|---------------|-------------|-------|----------|
| `PreTrainedTokenizer` | Python-based tokenizer | Slower | Full Python customization |
| `PreTrainedTokenizerFast` | Rust-based tokenizer | 10-100x faster | Offset mapping, batching |

### Key Tokenizer Methods

| Method | Purpose | Returns |
|--------|---------|--------|
| `tokenize()` | Split text into tokens | List of strings |
| `encode()` | Convert text to IDs | List of integers |
| `decode()` | Convert IDs back to text | String |
| `__call__()` | Full encoding with attention masks | Dictionary with tensors |
| `batch_encode_plus()` | Encode multiple texts | Batched dictionary |

In [ ]:
# ============================================================
# TOKENIZER METHODS DEMONSTRATION
# ============================================================

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Hugging Face transformers are amazing!"

print("="*60)
print("TOKENIZER METHODS")
print("="*60)

# Method 1: tokenize() - Returns token strings
tokens = tokenizer.tokenize(text)
print(f"\n1. tokenize('{text}')")
print(f"   Result: {tokens}")

# Method 2: encode() - Returns token IDs
ids = tokenizer.encode(text)
print(f"\n2. encode('{text}')")
print(f"   Result: {ids}")
print(f"   Note: Includes [CLS]={ids[0]} and [SEP]={ids[-1]}")

# Method 3: decode() - Convert IDs back to text
decoded = tokenizer.decode(ids)
print(f"\n3. decode({ids[:5]}...)")
print(f"   Result: '{decoded}'")

# Method 4: __call__() - Full encoding
encoded = tokenizer(text, return_tensors="pt")
print(f"\n4. tokenizer('{text}', return_tensors='pt')")
print(f"   Keys: {list(encoded.keys())}")
print(f"   input_ids shape: {encoded['input_ids'].shape}")

# Method 5: Batch encoding
texts = ["First sentence.", "Second sentence is longer."]
batch = tokenizer(texts, padding=True, return_tensors="pt")
print(f"\n5. Batch encoding of {len(texts)} texts")
print(f"   Batch shape: {batch['input_ids'].shape}")

TOKENIZER METHODS

1. tokenize('Hugging Face transformers are amazing!')
   Result: ['hugging', 'face', 'transformers', 'are', 'amazing', '!']

2. encode('Hugging Face transformers are amazing!')
   Result: [101, 17662, 2227, 19081, 2024, 6429, 999, 102]
   Note: Includes [CLS]=101 and [SEP]=102

3. decode([101, 17662, 2227, 19081, 2024]...)
   Result: '[CLS] hugging face transformers are amazing! [SEP]'

4. tokenizer('Hugging Face transformers are amazing!', return_tensors='pt')
   Keys: ['input_ids', 'token_type_ids', 'attention_mask']
   input_ids shape: torch.Size([1, 8])

5. Batch encoding of 2 texts
   Batch shape: torch.Size([2, 7])


## 1.4 Configuration Classes

Configuration classes store all the hyperparameters of a model. They can be used to:

1. **Inspect model settings**: See hidden_size, num_layers, etc.
2. **Modify before loading**: Change architecture before instantiation
3. **Create custom models**: Initialize from scratch with specific settings

In [ ]:
# ============================================================
# CONFIGURATION CLASSES
# ============================================================

from transformers import BertConfig, GPT2Config

print("="*60)
print("CONFIGURATION CLASSES COMPARISON")
print("="*60)

# BERT Configuration
bert_config = BertConfig()
print("\nDefault BertConfig:")
print(f"  vocab_size: {bert_config.vocab_size}")
print(f"  hidden_size: {bert_config.hidden_size}")
print(f"  num_hidden_layers: {bert_config.num_hidden_layers}")
print(f"  num_attention_heads: {bert_config.num_attention_heads}")
print(f"  intermediate_size: {bert_config.intermediate_size}")

# GPT-2 Configuration
gpt2_config = GPT2Config()
print("\nDefault GPT2Config:")
print(f"  vocab_size: {gpt2_config.vocab_size}")
print(f"  n_embd (hidden_size): {gpt2_config.n_embd}")
print(f"  n_layer: {gpt2_config.n_layer}")
print(f"  n_head: {gpt2_config.n_head}")

# Create a custom configuration
print("\nCreating Custom BERT Config (smaller model):")
custom_config = BertConfig(
    vocab_size=30522,
    hidden_size=256,        # Smaller than default 768
    num_hidden_layers=4,    # Fewer layers
    num_attention_heads=4,  # Fewer heads
    intermediate_size=1024  # Smaller FFN
)
print(f"  hidden_size: {custom_config.hidden_size}")
print(f"  num_hidden_layers: {custom_config.num_hidden_layers}")

CONFIGURATION CLASSES COMPARISON

Default BertConfig:
  vocab_size: 30522
  hidden_size: 768
  num_hidden_layers: 12
  num_attention_heads: 12
  intermediate_size: 3072

Default GPT2Config:
  vocab_size: 50257
  n_embd (hidden_size): 768
  n_layer: 12
  n_head: 12

Creating Custom BERT Config (smaller model):
  hidden_size: 256
  num_hidden_layers: 4


## 1.5 Training Classes (Trainer API)

The Trainer API provides a complete training loop with:

| Class | Purpose |
|-------|--------|
| `Trainer` | Complete training/evaluation loop |
| `TrainingArguments` | All training hyperparameters |
| `DataCollatorWithPadding` | Dynamic padding for batches |
| `DataCollatorForLanguageModeling` | MLM/CLM data preparation |

In [ ]:
# ============================================================
# TRAINER API OVERVIEW (Not executing - just demonstration)
# ============================================================

from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

print("="*60)
print("TRAINER API - KEY COMPONENTS")
print("="*60)

# TrainingArguments - All training settings
print("\nTrainingArguments key parameters:")
training_args_params = {
    "output_dir": "Path to save model checkpoints",
    "num_train_epochs": "Number of training epochs",
    "per_device_train_batch_size": "Batch size per GPU/CPU",
    "learning_rate": "Initial learning rate",
    "weight_decay": "L2 regularization",
    "warmup_steps": "Learning rate warmup steps",
    "logging_steps": "Log every N steps",
    "evaluation_strategy": "When to evaluate (steps/epoch)",
    "save_strategy": "When to save checkpoints",
    "fp16": "Use mixed precision training"
}

for param, desc in training_args_params.items():
    print(f"  {param}: {desc}")

# Example TrainingArguments (not executed)
print("\nExample TrainingArguments:")
print("""
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=100,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    fp16=True  # Enable if GPU supports it
)
""")

TRAINER API - KEY COMPONENTS

TrainingArguments key parameters:
  output_dir: Path to save model checkpoints
  num_train_epochs: Number of training epochs
  per_device_train_batch_size: Batch size per GPU/CPU
  learning_rate: Initial learning rate
  weight_decay: L2 regularization
  warmup_steps: Learning rate warmup steps
  logging_steps: Log every N steps
  evaluation_strategy: When to evaluate (steps/epoch)
  save_strategy: When to save checkpoints
  fp16: Use mixed precision training

Example TrainingArguments:

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=100,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    fp16=True  # Enable if GPU supports it
)



## 1.6 Summary: When to Use Which Class

| Task | Classes to Use |
|------|---------------|
| **Quick inference** | `pipeline()` |
| **Text classification** | `AutoTokenizer` + `AutoModelForSequenceClassification` |
| **NER/Token classification** | `AutoTokenizer` + `AutoModelForTokenClassification` |
| **Question Answering** | `AutoTokenizer` + `AutoModelForQuestionAnswering` |
| **Text Generation** | `AutoTokenizer` + `AutoModelForCausalLM` |
| **Translation/Summarization** | `AutoTokenizer` + `AutoModelForSeq2SeqLM` |
| **Feature extraction** | `AutoTokenizer` + `AutoModel` |
| **Fine-tuning** | Above + `Trainer` + `TrainingArguments` |

---

# Part 2: Getting Started with Pipelines

## What are Pipelines?

**Pipelines** are Hugging Face's highest-level API. They abstract away all the complexity of:
- Loading a pre-trained model
- Loading the corresponding tokenizer
- Preprocessing input text
- Running inference
- Post-processing outputs

With just **one line of code**, you can perform complex NLP tasks!

### Available Pipeline Tasks

| Task | Description | Example Use Case |
|------|-------------|------------------|
| `sentiment-analysis` | Classify text as positive/negative | Product reviews |
| `zero-shot-classification` | Classify text into custom labels | Content categorization |
| `text-generation` | Generate text continuations | Creative writing |
| `ner` | Extract named entities | Information extraction |
| `question-answering` | Answer questions from context | Document QA |
| `summarization` | Summarize long texts | News summarization |
| `translation` | Translate between languages | Multilingual apps |

---

## 2.1 Sentiment Analysis Pipeline

In [ ]:
# ============================================================
# SENTIMENT ANALYSIS
# ============================================================

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    device=0 if torch.cuda.is_available() else -1
)

print("Sentiment analyzer loaded!")
print(f"Model: {sentiment_analyzer.model.config._name_or_path}")

# Test sentences
sentences = [
    "I absolutely love this product! It exceeded all my expectations.",
    "This is the worst experience I've ever had.",
    "The food was good but the service was slow.",
    "It's okay, nothing special."
]

print("\n" + "="*60)
print("SENTIMENT ANALYSIS RESULTS")
print("="*60 + "\n")

for sentence in sentences:
    result = sentiment_analyzer(sentence)
    prediction = result[0]
    print(f"Text: \"{sentence[:50]}{'...' if len(sentence) > 50 else ''}\"")
    print(f"Result: {prediction['label']} (Confidence: {prediction['score']*100:.1f}%)")
    print("-" * 60)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentiment analyzer loaded!
Model: distilbert/distilbert-base-uncased-finetuned-sst-2-english

SENTIMENT ANALYSIS RESULTS

Text: "I absolutely love this product! It exceeded all my..."
Result: POSITIVE (Confidence: 100.0%)
------------------------------------------------------------
Text: "This is the worst experience I've ever had."
Result: NEGATIVE (Confidence: 100.0%)
------------------------------------------------------------
Text: "The food was good but the service was slow."
Result: NEGATIVE (Confidence: 99.7%)
------------------------------------------------------------
Text: "It's okay, nothing special."
Result: NEGATIVE (Confidence: 81.9%)
------------------------------------------------------------


## 2.2 Zero-Shot Classification Pipeline

In [ ]:
# ============================================================
# ZERO-SHOT CLASSIFICATION
# ============================================================

zero_shot = pipeline(
    "zero-shot-classification",
    device=0 if torch.cuda.is_available() else -1
)

# Define custom labels
labels = ["technology", "sports", "politics", "entertainment"]

texts = [
    "Apple announces new MacBook Pro with M3 chip",
    "Manchester United wins Champions League",
    "Senate passes new infrastructure bill"
]

print("="*60)
print("ZERO-SHOT CLASSIFICATION")
print(f"Labels: {labels}")
print("="*60 + "\n")

for text in texts:
    result = zero_shot(text, candidate_labels=labels)
    print(f"Text: \"{text}\"")
    print(f"Top: {result['labels'][0]} ({result['scores'][0]*100:.1f}%)")
    print()

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

ZERO-SHOT CLASSIFICATION
Labels: ['technology', 'sports', 'politics', 'entertainment']

Text: "Apple announces new MacBook Pro with M3 chip"
Top: technology (92.3%)

Text: "Manchester United wins Champions League"
Top: sports (96.7%)

Text: "Senate passes new infrastructure bill"
Top: technology (31.7%)



## 2.3 Text Generation Pipeline

### Decoding Strategies

| Strategy | Description | Use Case |
|----------|-------------|----------|
| **Greedy** | Always pick highest probability | Fast, deterministic |
| **Sampling** | Random based on probabilities | Creative, diverse |
| **Top-K** | Sample from top K tokens | Balanced |
| **Top-P (Nucleus)** | Sample from cumulative prob >= P | Adaptive |
| **Beam Search** | Keep N best sequences | High quality |


| Parameter | Controls    | Low Value     | High Value      |
| --------- | ----------- | ------------- | --------------- |
| p         | Prob mass   | Focused (0.7) | Creative (0.95) |
| k         | Token count | Focused (20)  | Creative (100)  |

#### Transformer Generation Strategies: **p, k, & Beam Search**

#### **1. TOP-P SAMPLING** (`top_p=0.9`)

**What is `p`?** Cumulative probability threshold

```python
# Model predicts: ["the":0.4, "a":0.3, "cat":0.15, "dog":0.08, ...]
# top_p=0.9 → Keep tokens until cumulative ≥ 90%
0.4 + 0.3 + 0.15 + 0.08 = 0.93 ✓ → Sample from these 4 tokens only
```

| **p Value** | **Effect** | **Use Case** |
|-------------|------------|--------------|
| `0.7` | Focused | Professional writing |
| `0.9` | **Balanced**  | Chatbots |
| `0.95` | Creative | Storytelling |

**Code**:
```python
output = generator(prompt, do_sample=True, top_p=0.9, temperature=0.8)
```

---

#### **2. TOP-K SAMPLING** (`top_k=50`)

**What is `k`?** Fixed number of top tokens

```python
# Regardless of probabilities, always sample from TOP 50 tokens
# ["the":0.4, "a":0.3, ...] → Keep highest 50 → Sample randomly
```

| **k Value** | **Effect** | **Use Case** |
|-------------|------------|--------------|
| `20` | Very focused | Technical |
| `50` | **Balanced**  | General |
| `100` | Creative | Fiction |

**Code**:
```python
output = generator(prompt, do_sample=True, top_k=50, temperature=0.8)
```

---

#### **3. BEAM SEARCH** (`num_beams=4`)

**What is beam?** Number of parallel sequences


In [ ]:
# ============================================================
# TEXT GENERATION WITH DIFFERENT STRATEGIES
# ============================================================

text_generator = pipeline(
    "text-generation",
    model="gpt2",
    device=0 if torch.cuda.is_available() else -1
)

prompt = "The future of artificial intelligence is"

print("="*60)
print("TEXT GENERATION STRATEGIES")
print(f"Prompt: \"{prompt}\"")
print("="*60)

# Greedy
print("\n1. GREEDY DECODING:")
output = text_generator(prompt, max_length=50, do_sample=False, pad_token_id=50256)
print(f"   {output[0]['generated_text'][len(prompt):].strip()[:100]}...")

# Top-P Sampling
print("\n2. TOP-P SAMPLING (p=0.9, temp=0.8):")
output = text_generator(prompt, max_length=50, do_sample=True, top_p=0.9, temperature=0.8, pad_token_id=50256)
print(f"   {output[0]['generated_text'][len(prompt):].strip()[:100]}...")

# Beam Search
print("\n3. BEAM SEARCH (num_beams=4):")
output = text_generator(prompt, max_length=50, num_beams=4, do_sample=False, pad_token_id=50256)
print(f"   {output[0]['generated_text'][len(prompt):].strip()[:100]}...")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEXT GENERATION STRATEGIES
Prompt: "The future of artificial intelligence is"

1. GREEDY DECODING:


Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'temperature', 'top_p', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   uncertain.

"We're not sure what the future will look like," said Dr. Michael S. Schoenfeld, a profe...

2. TOP-P SAMPLING (p=0.9, temp=0.8):


Passing `generation_config` together with generation-related arguments=({'do_sample', 'num_beams', 'pad_token_id', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   unknown."

"It's really difficult to say," said CTO of Microsoft's AI division, Mark Shuttleworth. "...

3. BEAM SEARCH (num_beams=4):
   in the hands of the next generation of researchers.

"We're going to see a lot of advances in artifi...


## 2.4 Named Entity Recognition (NER) Pipeline

Input: "John works at Google"

BERT splits → ['John', '▁work', 's', '▁at', '▁Google']

NER labels each token:
['B-PER', 'B-ORG', 'I-ORG', 'O', 'B-ORG'] </br>
  ↑ </br>
"John" = 1 token </br>

B-PER = Beginning of Person entity </br>
I-PER = Inside Person entity       </br>
O     = Outside any entity         </br>



In [ ]:
# ============================================================
# NAMED ENTITY RECOGNITION
# ============================================================

from collections import defaultdict

ner = pipeline(
    "ner",
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

text = """Apple CEO Tim Cook announced a partnership with Microsoft
in San Francisco. Goldman Sachs predicts this could boost Apple's
market value significantly."""

print("="*60)
print("NAMED ENTITY RECOGNITION")
print("="*60)
print(f"\nText: {text}\n")

entities = ner(text)

# Group by type
by_type = defaultdict(list)
for ent in entities:
    by_type[ent['entity_group']].append(f"{ent['word']} ({ent['score']*100:.0f}%)")

print("Extracted Entities:")
for etype, words in by_type.items():
    print(f"  {etype}: {', '.join(words)}")

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

NAMED ENTITY RECOGNITION

Text: Apple CEO Tim Cook announced a partnership with Microsoft 
in San Francisco. Goldman Sachs predicts this could boost Apple's 
market value significantly.

Extracted Entities:
  ORG: Apple (100%), Microsoft (100%), Goldman Sachs (100%), Apple (100%)
  PER: Tim Cook (100%)
  LOC: San Francisco (100%)


## 2.5 Question Answering Pipeline

In [ ]:
# ============================================================
# QUESTION ANSWERING
# ============================================================

qa = pipeline(
    "question-answering",
    device=0 if torch.cuda.is_available() else -1
)

context = """
Climate change is one of the most pressing challenges facing humanity.
The Earth's average temperature has increased by approximately 1.1 degrees
Celsius since the pre-industrial era. The IPCC warns that if global
temperatures rise beyond 1.5 degrees Celsius, we may face catastrophic
consequences. To limit warming to 1.5 degrees, global carbon emissions
must be reduced by 45% by 2030.
"""

questions = [
    "How much has the Earth's temperature increased?",
    "What organization warns about temperature rise?",
    "By what percentage must emissions be reduced by 2030?"
]

print("="*60)
print("QUESTION ANSWERING")
print("="*60 + "\n")

for q in questions:
    result = qa(question=q, context=context)
    print(f"Q: {q}")
    print(f"A: {result['answer']} (confidence: {result['score']*100:.1f}%)")
    print()

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

QUESTION ANSWERING

Q: How much has the Earth's temperature increased?
A: 1.1 degrees 
Celsius (confidence: 48.8%)

Q: What organization warns about temperature rise?
A: The IPCC (confidence: 62.5%)

Q: By what percentage must emissions be reduced by 2030?
A: 45% (confidence: 74.5%)



---

# Part 3: Working with Tokenizers

## What are Tokenizers?

**Tokenizers** convert text into a format that models can understand:

1. **Split text** into smaller units (tokens)
2. **Map tokens** to numerical IDs
3. **Add special tokens** required by the model
4. **Handle padding/truncation** for batched inputs

### Subword Algorithms

| Algorithm | Used By | Key Feature |
|-----------|---------|-------------|
| **WordPiece** | BERT, DistilBERT | Uses `##` prefix for subwords |
| **BPE** | GPT-2, RoBERTa | Iteratively merges frequent pairs |
| **SentencePiece** | T5, ALBERT | Language-agnostic |

---

## 3.1 Comparing BERT vs GPT-2 Tokenizers

In [ ]:
# ============================================================
# BERT vs GPT-2 TOKENIZER COMPARISON
# ============================================================

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

print("="*60)
print("TOKENIZER COMPARISON: BERT vs GPT-2")
print("="*60)

print(f"\nBERT: vocab_size={bert_tok.vocab_size}, algorithm=WordPiece")
print(f"GPT-2: vocab_size={gpt2_tok.vocab_size}, algorithm=BPE")

texts = [
    "Hello, world!",
    "Machine learning",
    "unhappiness",
    "ChatGPT"
]

print("\nTokenization Comparison:")
print("-" * 60)

for text in texts:
    bert_tokens = bert_tok.tokenize(text)
    gpt2_tokens = gpt2_tok.tokenize(text)
    print(f"\n'{text}'")
    print(f"  BERT:  {bert_tokens}")
    print(f"  GPT-2: {gpt2_tokens}")

TOKENIZER COMPARISON: BERT vs GPT-2

BERT: vocab_size=30522, algorithm=WordPiece
GPT-2: vocab_size=50257, algorithm=BPE

Tokenization Comparison:
------------------------------------------------------------

'Hello, world!'
  BERT:  ['hello', ',', 'world', '!']
  GPT-2: ['Hello', ',', 'Ġworld', '!']

'Machine learning'
  BERT:  ['machine', 'learning']
  GPT-2: ['Machine', 'Ġlearning']

'unhappiness'
  BERT:  ['un', '##ha', '##pp', '##iness']
  GPT-2: ['un', 'h', 'appiness']

'ChatGPT'
  BERT:  ['chat', '##gp', '##t']
  GPT-2: ['Chat', 'G', 'PT']


## 3.2 Padding and Truncation

SyntaxError: f-string: unmatched ')' (3754265083.py, line 31)

In [ ]:
# ============================================================
# PADDING AND TRUNCATION -  EXPLANATION
# ============================================================

sentences = [
    "Hi there!",                                # Short: 3 words
    "The weather is nice today.",              # Medium: 5 words
    "Machine learning models require substantial computational resources."  # Long: 7+ words
]

print("="*70)
print("WHY BERT NEEDS SAME LENGTH SENTENCES")
print("="*70)

# 1. SHOW THE PROBLEM - Different lengths = BERT CRASH!
print("\n1. WITHOUT PADDING (Different lengths = BAD!)")
token_lengths = []
for i, s in enumerate(sentences):
    ids = bert_tok.encode(s)
    token_lengths.append(len(ids))
    print(f"   Sentence {i+1}: '{s}' → {len(ids)} tokens")

print(f"\n   PROBLEM: Lengths = {token_lengths} → BERT wants [12,12,12]!")

# 2. MAGIC FIX #1 - PAD TO LONGEST (Efficient!)
print("\n2. PADDING='longest' (Pad shorts to match longest)")
padded = bert_tok(sentences, padding="longest", return_tensors="pt")

print(f"   ALL SAME LENGTH: {padded['input_ids'].shape}")
print(f"   3 sentences, {padded['input_ids'].shape[1]} tokens each!")
print(f"   Short sentences got {max(token_lengths)} - {min(token_lengths)} = {max(token_lengths)-min(token_lengths)} padding tokens")

# 3. MAGIC FIX #2 - FIXED SIZE (Predictable!)
print("\n3. TRUNCATION + PADDING='max_length' (Fixed size box)")
truncated = bert_tok(sentences, max_length=10, truncation=True, padding="max_length", return_tensors="pt")
print(f"   FIXED BOX: {truncated['input_ids'].shape} (always [3,10])")
print(f"   Long sentences CUT, short ones PADDED to exactly 10 tokens")

# 4. SHOW ACTUAL TOKENS (What padding looks like!)
print("\n4. SEE THE PADDING TOKENS (0 = [PAD])")
for i, s in enumerate(sentences):
    tokens = padded['input_ids'][i].tolist()
    mask = padded['attention_mask'][i].tolist()
    real_tokens = sum(mask)
    pad_tokens = len(tokens) - real_tokens

    print(f"\n   Sentence {i+1}: '{s}'")
    print(f"      Tokens:  {tokens[:15]}{'...' if len(tokens)>15 else ''}")
    print(f"      Mask:    {mask[:15]}{'...' if len(mask)>15 else ''}")
    print(f"      {real_tokens} REAL words + {pad_tokens} [PAD] tokens (0s)")
    print(f"      BERT ignores 0s thanks to mask!")

# 5. VISUAL SUMMARY
print("\n5. BEFORE vs AFTER")
print("   BEFORE:  [4 tokens]  [7 tokens]  [12 tokens]  → CRASH!")
print("   AFTER:   [12]        [12]        [12]         → WORKS!")
print("\n   MASK:    [1,1,1,1,0,0,0,0...] → 'Focus here, ignore padding!'")

WHY BERT NEEDS SAME LENGTH SENTENCES

1. WITHOUT PADDING (Different lengths = BAD!)
   Sentence 1: 'Hi there!' → 5 tokens
   Sentence 2: 'The weather is nice today.' → 8 tokens
   Sentence 3: 'Machine learning models require substantial computational resources.' → 10 tokens

   PROBLEM: Lengths = [5, 8, 10] → BERT wants [12,12,12]!

2. PADDING='longest' (Pad shorts to match longest)
   ALL SAME LENGTH: torch.Size([3, 10])
   3 sentences, 10 tokens each!
   Short sentences got 10 - 5 = 5 padding tokens

3. TRUNCATION + PADDING='max_length' (Fixed size box)
   FIXED BOX: torch.Size([3, 10]) (always [3,10])
   Long sentences CUT, short ones PADDED to exactly 10 tokens

4. SEE THE PADDING TOKENS (0 = [PAD])

   Sentence 1: 'Hi there!'
      Tokens:  [101, 7632, 2045, 999, 102, 0, 0, 0, 0, 0]
      Mask:    [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
      5 REAL words + 5 [PAD] tokens (0s)
      BERT ignores 0s thanks to mask!

   Sentence 2: 'The weather is nice today.'
      Tokens:  [101, 1996, 4633

## 3.3 Adding Custom Tokens

In [ ]:
# ============================================================
# ADDING CUSTOM TOKENS
# ============================================================

from transformers import AutoTokenizer, AutoModel

# Fresh tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

print("="*60)
print("ADDING CUSTOM TOKENS")
print("="*60)

print(f"\nBefore: vocab_size = {len(tokenizer)}")

# Check how terms are tokenized before adding
print("\nBefore adding (split into subwords):")
for term in ["[USER]", "ChatGPT", "LangChain"]:
    tokens = tokenizer.tokenize(term)
    print(f"  '{term}' -> {tokens}")

# Add special tokens
special_tokens = {'additional_special_tokens': ['[USER]', '[ASSISTANT]']}
tokenizer.add_special_tokens(special_tokens)

# Add regular tokens
tokenizer.add_tokens(['ChatGPT', 'LangChain', 'huggingface'])

print(f"\nAfter: vocab_size = {len(tokenizer)}")

# Verify single tokens now
print("\nAfter adding (single tokens):")
for term in ["[USER]", "ChatGPT", "LangChain"]:
    tokens = tokenizer.tokenize(term)
    print(f"  '{term}' -> {tokens}")

# CRITICAL: Resize model embeddings
print(f"\nModel embedding size before: {model.embeddings.word_embeddings.weight.shape}")
model.resize_token_embeddings(len(tokenizer))
print(f"Model embedding size after: {model.embeddings.word_embeddings.weight.shape}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ADDING CUSTOM TOKENS

Before: vocab_size = 30522

Before adding (split into subwords):
  '[USER]' -> ['[', 'user', ']']
  'ChatGPT' -> ['chat', '##gp', '##t']
  'LangChain' -> ['lang', '##chai', '##n']

After: vocab_size = 30527

After adding (single tokens):
  '[USER]' -> ['[USER]']
  'ChatGPT' -> ['chatgpt']
  'LangChain' -> ['langchain']

Model embedding size before: torch.Size([30522, 768])


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model embedding size after: torch.Size([30527, 768])


---

# Summary and Key Takeaways

## What We Covered

### Part 1: Important Classes
- **Auto Classes**: AutoTokenizer, AutoModel, AutoConfig, AutoModelForXXX
- **Tokenizer Classes**: PreTrainedTokenizer, PreTrainedTokenizerFast
- **Configuration Classes**: BertConfig, GPT2Config, etc.
- **Training Classes**: Trainer, TrainingArguments

### Part 2: Pipelines
- **Sentiment Analysis**: Classifying text emotions
- **Zero-Shot Classification**: Custom labels without fine-tuning
- **Text Generation**: Various decoding strategies
- **NER**: Extracting named entities
- **Question Answering**: Extractive QA from context

### Part 3: Tokenizers
- **Basic tokenization**: encode, decode, special tokens
- **Comparing tokenizers**: BERT (WordPiece) vs GPT-2 (BPE)
- **Padding and truncation**: Batch processing
- **Custom tokens**: Adding domain-specific vocabulary

## Quick Reference: Class Selection Guide

| Your Task | Start With |
|-----------|------------|
| Quick prototype | `pipeline()` |
| Custom inference | `AutoTokenizer` + `AutoModelForXXX` |
| Fine-tuning | Above + `Trainer` + `TrainingArguments` |
| Feature extraction | `AutoTokenizer` + `AutoModel` |

## Next Steps

1. **Fine-tune a model**: Use the Trainer API for your specific task
2. **Explore more pipelines**: Try summarization, translation, fill-mask
3. **Work with datasets**: Use the `datasets` library
4. **Deploy models**: Learn about model optimization

## Resources

- [Hugging Face Documentation](https://huggingface.co/docs)
- [Hugging Face Course](https://huggingface.co/course)
- [Model Hub](https://huggingface.co/models)
- [Transformers GitHub](https://github.com/huggingface/transformers)

---